# Fine-Tuning Sentence Transformers for Resume–Job Matching

**What this notebook does:**
1. Loads your 500 curated resume-JD training pairs
2. Evaluates 3 base models (before fine-tuning)
3. Fine-tunes each model with CoSENTLoss
4. Evaluates fine-tuned models (after)
5. Compares results with visualization
6. Pushes best model to HuggingFace Hub

**Runtime:** Select **GPU → T4** in Colab (Runtime → Change runtime type)

**Time:** ~45 minutes total (including training all 3 models)


## Cell 1: Install dependencies

In [ ]:
# Install required packages
!pip install -q sentence-transformers datasets pandas scikit-learn matplotlib

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ No GPU detected! Go to Runtime → Change runtime type → T4 GPU")


## Cell 2: Load your training data

Upload `resume_jd_training_data_500.csv` to Colab, or mount Google Drive.


In [ ]:
import pandas as pd
import numpy as np
from collections import Counter

# Option 1: Upload directly
from google.colab import files
uploaded = files.upload()  # Click "Choose Files" and select your CSV

# Get the filename (handles any name)
filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)

# Option 2: If you uploaded to Google Drive instead, uncomment these:
# from google.colab import drive
# drive.mount('/content/drive')
# df = pd.read_csv('/content/drive/MyDrive/resume_jd_training_data_500.csv')

print(f"Loaded {len(df)} training pairs")
print(f"\nColumns: {list(df.columns)}")
print(f"\nMatch type distribution:")
print(df['match_type'].value_counts())
print(f"\nScore statistics:")
print(df['score'].describe())
print(f"\nIndustry coverage: {df['industry'].nunique()} industries")
print(f"\nSample pair:")
sample = df.iloc[0]
print(f"  Score: {sample['score']}")
print(f"  Type: {sample['match_type']}")
print(f"  Resume: {sample['resume'][:150]}...")
print(f"  JD: {sample['jd'][:150]}...")


## Cell 3: Prepare train / validation / test splits

**Split strategy:** 80% train (400), 10% validation (50), 10% test (50)

Validation is used during training to track progress.
Test is held out completely — only used for final evaluation.


In [ ]:
from sentence_transformers import InputExample
from torch.utils.data import DataLoader

# Stratified split by match_type to keep distribution balanced across splits
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['match_type']
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, random_state=42, stratify=temp_df['match_type']
)

print(f"Train: {len(train_df)} pairs")
print(f"  {dict(Counter(train_df['match_type']))}")
print(f"\nValidation: {len(val_df)} pairs")
print(f"  {dict(Counter(val_df['match_type']))}")
print(f"\nTest: {len(test_df)} pairs (held out)")
print(f"  {dict(Counter(test_df['match_type']))}")

# Convert to sentence-transformers InputExample format
train_examples = [
    InputExample(
        texts=[row['resume'], row['jd']],
        label=float(row['score'])
    )
    for _, row in train_df.iterrows()
]

print(f"\n✅ Created {len(train_examples)} training examples")


## Cell 4: Evaluate base models (BEFORE fine-tuning)

This measures how well the pre-trained models score resume-JD pairs
**without any domain-specific training**. These are your baselines.


In [ ]:
from sentence_transformers import SentenceTransformer
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator
from scipy.stats import spearmanr
from sklearn.metrics.pairwise import cosine_similarity
import time

# The 3 models we'll compare
MODEL_NAMES = {
    'MiniLM': 'all-MiniLM-L6-v2',
    'MPNet': 'all-mpnet-base-v2',
    'BGE': 'BAAI/bge-small-en-v1.5',
}

def evaluate_model(model, eval_df, label=""):
    """Evaluate a model on a dataframe with resume, jd, score columns."""
    resumes = eval_df['resume'].tolist()
    jds = eval_df['jd'].tolist()
    true_scores = eval_df['score'].tolist()

    # Encode
    start = time.time()
    resume_embs = model.encode(resumes, show_progress_bar=False)
    jd_embs = model.encode(jds, show_progress_bar=False)
    encode_time = time.time() - start

    # Predict similarity
    predicted = [
        cosine_similarity([r], [j])[0][0]
        for r, j in zip(resume_embs, jd_embs)
    ]

    # Metrics
    spearman, _ = spearmanr(true_scores, predicted)
    mae = np.mean(np.abs(np.array(true_scores) - np.array(predicted)))

    return {
        'spearman': spearman,
        'mae': mae,
        'encode_time': encode_time,
        'predictions': predicted,
    }

# Evaluate all 3 base models
base_results = {}

for name, model_id in MODEL_NAMES.items():
    print(f"\nEvaluating base {name}...")
    model = SentenceTransformer(model_id)
    results = evaluate_model(model, test_df, f"base_{name}")
    base_results[name] = results
    print(f"  Spearman: {results['spearman']:.4f}")
    print(f"  MAE:      {results['mae']:.4f}")
    print(f"  Speed:    {results['encode_time']:.1f}s for {len(test_df)*2} texts")
    del model
    torch.cuda.empty_cache()

print(f"\n{'='*50}")
print("BASE MODEL COMPARISON (before fine-tuning)")
print(f"{'='*50}")
print(f"{'Model':<10} {'Spearman':>10} {'MAE':>10} {'Speed':>10}")
print("-" * 42)
for name in MODEL_NAMES:
    r = base_results[name]
    print(f"{name:<10} {r['spearman']:>10.4f} {r['mae']:>10.4f} {r['encode_time']:>8.1f}s")


## Cell 5: Fine-tune MiniLM (fastest, ~8 min)

**Why MiniLM first:** It's the smallest model (22M params), so it trains fastest.
Good for verifying your pipeline works before investing time in larger models.


In [ ]:
from sentence_transformers import SentenceTransformer, losses

# Load fresh base model
model_minilm = SentenceTransformer('all-MiniLM-L6-v2')

# DataLoader
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=32)

# Loss function: CoSENTLoss
# Why CoSENTLoss over CosineSimilarityLoss?
# - Optimizes RANKING (relative order) not absolute scores
# - Stronger gradients = faster convergence
# - Better for retrieval tasks like "find the best resume for this JD"
train_loss = losses.CoSENTLoss(model=model_minilm)

# Evaluator (runs after each epoch)
evaluator = EmbeddingSimilarityEvaluator(
    sentences1=val_df['resume'].tolist(),
    sentences2=val_df['jd'].tolist(),
    scores=val_df['score'].tolist(),
    name='resume-jd-val',
)

# Train!
print("Fine-tuning MiniLM...")
print(f"  Epochs: 4")
print(f"  Batch size: 32")
print(f"  Training examples: {len(train_examples)}")
print(f"  Steps per epoch: {len(train_dataloader)}")

model_minilm.fit(
    train_objectives=[(train_dataloader, train_loss)],
    evaluator=evaluator,
    epochs=4,
    warmup_steps=int(len(train_dataloader) * 0.1),
    evaluation_steps=len(train_dataloader),  # Evaluate every epoch
    output_path="models/minilm-resume-matcher",
    show_progress_bar=True,
    use_amp=True,  # Mixed precision for speed on T4
)

print("\n✅ MiniLM fine-tuning complete!")
print("   Saved to: models/minilm-resume-matcher")


## Cell 6: Fine-tune MPNet (best accuracy, ~15 min)

**MPNet** is the gold standard sentence transformer. 109M parameters, 768-dim embeddings.
This will likely be your best-performing model.


In [ ]:
# Load fresh base model
model_mpnet = SentenceTransformer('all-mpnet-base-v2')

# DataLoader (recreate for fresh shuffle)
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=16)  # Smaller batch for larger model

# Loss
train_loss = losses.CoSENTLoss(model=model_mpnet)

# Evaluator
evaluator = EmbeddingSimilarityEvaluator(
    sentences1=val_df['resume'].tolist(),
    sentences2=val_df['jd'].tolist(),
    scores=val_df['score'].tolist(),
    name='resume-jd-val',
)

print("Fine-tuning MPNet...")
print(f"  Epochs: 4")
print(f"  Batch size: 16 (smaller for 109M param model)")

model_mpnet.fit(
    train_objectives=[(train_dataloader, train_loss)],
    evaluator=evaluator,
    epochs=4,
    warmup_steps=int(len(train_dataloader) * 0.1),
    evaluation_steps=len(train_dataloader),
    output_path="models/mpnet-resume-matcher",
    show_progress_bar=True,
    use_amp=True,
)

print("\n✅ MPNet fine-tuning complete!")
print("   Saved to: models/mpnet-resume-matcher")


## Cell 7: Fine-tune BGE (retrieval specialist, ~10 min)

**BGE** was specifically designed for retrieval tasks. 33M params with 384-dim embeddings.
Interesting to compare against MiniLM (same embedding size, different architecture).


In [ ]:
# Load fresh base model
model_bge = SentenceTransformer('BAAI/bge-small-en-v1.5')

# DataLoader
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=32)

# Loss
train_loss = losses.CoSENTLoss(model=model_bge)

# Evaluator
evaluator = EmbeddingSimilarityEvaluator(
    sentences1=val_df['resume'].tolist(),
    sentences2=val_df['jd'].tolist(),
    scores=val_df['score'].tolist(),
    name='resume-jd-val',
)

print("Fine-tuning BGE...")
model_bge.fit(
    train_objectives=[(train_dataloader, train_loss)],
    evaluator=evaluator,
    epochs=4,
    warmup_steps=int(len(train_dataloader) * 0.1),
    evaluation_steps=len(train_dataloader),
    output_path="models/bge-resume-matcher",
    show_progress_bar=True,
    use_amp=True,
)

print("\n✅ BGE fine-tuning complete!")
print("   Saved to: models/bge-resume-matcher")


## Cell 8: Evaluate fine-tuned models & compare

This is the payoff — how much did fine-tuning improve each model?


In [ ]:
# Load fine-tuned models
ft_models = {
    'MiniLM': SentenceTransformer('models/minilm-resume-matcher'),
    'MPNet': SentenceTransformer('models/mpnet-resume-matcher'),
    'BGE': SentenceTransformer('models/bge-resume-matcher'),
}

# Evaluate on HELD-OUT test set
ft_results = {}
for name, model in ft_models.items():
    print(f"Evaluating fine-tuned {name}...")
    ft_results[name] = evaluate_model(model, test_df, f"ft_{name}")

# Comparison table
print(f"\n{'='*70}")
print("BEFORE vs AFTER FINE-TUNING (Test Set)")
print(f"{'='*70}")
print(f"{'Model':<10} {'Base Spear':>11} {'FT Spear':>10} {'Δ Spear':>9} {'Base MAE':>10} {'FT MAE':>8} {'Δ MAE':>7}")
print("-" * 67)

for name in MODEL_NAMES:
    bs = base_results[name]['spearman']
    fs = ft_results[name]['spearman']
    bm = base_results[name]['mae']
    fm = ft_results[name]['mae']
    ds = fs - bs
    dm = fm - bm
    print(f"{name:<10} {bs:>11.4f} {fs:>10.4f} {'+' if ds > 0 else ''}{ds:>8.4f} {bm:>10.4f} {fm:>8.4f} {dm:>7.4f}")

# Find best model
best_name = max(ft_results, key=lambda n: ft_results[n]['spearman'])
print(f"\n🏆 Best model: {best_name} (Spearman: {ft_results[best_name]['spearman']:.4f})")


## Cell 9: Visualize results

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Spearman comparison
models = list(MODEL_NAMES.keys())
base_spear = [base_results[m]['spearman'] for m in models]
ft_spear = [ft_results[m]['spearman'] for m in models]

x = np.arange(len(models))
axes[0].bar(x - 0.2, base_spear, 0.35, label='Base', color='#B4B2A9', edgecolor='white')
axes[0].bar(x + 0.2, ft_spear, 0.35, label='Fine-tuned', color='#185FA5', edgecolor='white')
axes[0].set_ylabel('Spearman Correlation')
axes[0].set_title('Spearman: Base vs Fine-tuned', fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(models)
axes[0].legend()
axes[0].set_ylim(0, 1)

# 2. MAE comparison
base_mae = [base_results[m]['mae'] for m in models]
ft_mae = [ft_results[m]['mae'] for m in models]

axes[1].bar(x - 0.2, base_mae, 0.35, label='Base', color='#B4B2A9', edgecolor='white')
axes[1].bar(x + 0.2, ft_mae, 0.35, label='Fine-tuned', color='#0F6E56', edgecolor='white')
axes[1].set_ylabel('Mean Absolute Error')
axes[1].set_title('MAE: Base vs Fine-tuned (lower is better)', fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels(models)
axes[1].legend()

# 3. Score distribution: predicted vs actual (best model)
best_preds = ft_results[best_name]['predictions']
true_scores = test_df['score'].tolist()

axes[2].scatter(true_scores, best_preds, alpha=0.6, s=30, c='#534AB7', edgecolors='white', linewidth=0.5)
axes[2].plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Perfect')
axes[2].set_xlabel('True Score')
axes[2].set_ylabel('Predicted Score')
axes[2].set_title(f'{best_name}: Predicted vs True Scores', fontweight='bold')
axes[2].legend()
axes[2].set_xlim(0, 1)
axes[2].set_ylim(0, 1)

plt.tight_layout()
plt.savefig('finetuning_results.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n📊 Chart saved: finetuning_results.png")


## Cell 10: Error analysis by match type

Where does the model succeed and where does it struggle?
This analysis directly informs what additional training data you'd need.


In [ ]:
# Use the best model
best_model = ft_models[best_name]
results = ft_results[best_name]

# Add predictions to test dataframe
test_analysis = test_df.copy()
test_analysis['predicted'] = results['predictions']
test_analysis['error'] = abs(test_analysis['score'] - test_analysis['predicted'])

# Error by match type
print(f"Error Analysis by Match Type ({best_name})")
print("=" * 60)
for mt in ['strong', 'good', 'partial', 'hard_negative', 'weak']:
    subset = test_analysis[test_analysis['match_type'] == mt]
    if len(subset) > 0:
        mae = subset['error'].mean()
        spear, _ = spearmanr(subset['score'], subset['predicted'])
        worst = subset.nlargest(1, 'error').iloc[0] if len(subset) > 0 else None
        print(f"\n  {mt}:")
        print(f"    Samples: {len(subset)}")
        print(f"    MAE: {mae:.4f}")
        print(f"    Spearman: {spear:.4f}" if len(subset) > 2 else "    Spearman: N/A (too few)")
        if worst is not None:
            print(f"    Worst error: {worst['error']:.3f} (true={worst['score']:.3f}, pred={worst['predicted']:.3f})")

# Error by industry
print(f"\n\nError Analysis by Industry")
print("=" * 60)
for ind in sorted(test_analysis['industry'].unique()):
    subset = test_analysis[test_analysis['industry'] == ind]
    if len(subset) > 0:
        mae = subset['error'].mean()
        print(f"  {ind:<45}: MAE={mae:.4f} (n={len(subset)})")

# Biggest errors (to understand failure modes)
print(f"\n\nTop 5 Biggest Errors")
print("=" * 60)
worst5 = test_analysis.nlargest(5, 'error')
for _, row in worst5.iterrows():
    print(f"\n  True: {row['score']:.3f} | Predicted: {row['predicted']:.3f} | Error: {row['error']:.3f}")
    print(f"  Type: {row['match_type']} | Industry: {row['industry']}")
    print(f"  Resume: {row['resume'][:100]}...")
    print(f"  JD: {row['jd'][:100]}...")


## Cell 11: Qualitative demo — score any resume against any JD

This is what you'd show in a portfolio presentation or interview.


In [ ]:
def score_match(resume_text, jd_text, model=None):
    """Score a resume against a job description. Returns 0-1 similarity."""
    if model is None:
        model = ft_models[best_name]

    resume_emb = model.encode([resume_text])
    jd_emb = model.encode([jd_text])
    score = cosine_similarity(resume_emb, jd_emb)[0][0]
    return float(score)

# Demo with a sample from test set
sample = test_df.iloc[0]

print("DEMO: Resume-JD Match Scoring")
print("=" * 50)
print(f"\nResume: {sample['resume'][:200]}...")
print(f"\nJD: {sample['jd'][:200]}...")
print(f"\nTrue score:      {sample['score']:.3f}")
print(f"Predicted score: {score_match(sample['resume'], sample['jd']):.3f}")
print(f"Match type:      {sample['match_type']}")

# Compare base vs fine-tuned on this sample
print(f"\n\nBase vs Fine-tuned on same pair:")
for name, model_id in MODEL_NAMES.items():
    base_model = SentenceTransformer(model_id)
    base_score = score_match(sample['resume'], sample['jd'], base_model)
    ft_score = score_match(sample['resume'], sample['jd'], ft_models[name])
    print(f"  {name}: Base={base_score:.3f} → Fine-tuned={ft_score:.3f} (true={sample['score']:.3f})")
    del base_model

# Try your own!
print(f"\n\n💡 Try your own resume-JD pair:")
print("   score = score_match(your_resume_text, your_jd_text)")
print("   print(f'Match score: {score:.3f}')")


## Cell 12: Push best model to HuggingFace Hub

This makes your fine-tuned model publicly available and downloadable.


In [ ]:
# Login to HuggingFace
from huggingface_hub import login

# Get your token from https://huggingface.co/settings/tokens
login()  # Will prompt for your token

# Push the best model
REPO_NAME = "resume-jd-matcher-mpnet"  # Change to your preferred name
USERNAME = "YOUR_HF_USERNAME"          # Change to your HuggingFace username

best_model = ft_models[best_name]
best_model.save_to_hub(
    f"{USERNAME}/{REPO_NAME}",
    exist_ok=True,
    train_datasets=["custom-resume-jd-500"],
)

print(f"\n✅ Model pushed to: https://huggingface.co/{USERNAME}/{REPO_NAME}")
print(f"\nAnyone can now use it:")
print(f"  from sentence_transformers import SentenceTransformer")
print(f"  model = SentenceTransformer('{USERNAME}/{REPO_NAME}')")

# Save results summary for your portfolio
results_summary = {
    'models_compared': list(MODEL_NAMES.keys()),
    'training_pairs': len(train_df),
    'test_pairs': len(test_df),
    'best_model': best_name,
    'results': {}
}

for name in MODEL_NAMES:
    results_summary['results'][name] = {
        'base_spearman': round(base_results[name]['spearman'], 4),
        'ft_spearman': round(ft_results[name]['spearman'], 4),
        'improvement': round(ft_results[name]['spearman'] - base_results[name]['spearman'], 4),
        'base_mae': round(base_results[name]['mae'], 4),
        'ft_mae': round(ft_results[name]['mae'], 4),
    }

import json
with open('training_results.json', 'w') as f:
    json.dump(results_summary, f, indent=2)

print(f"\n📊 Results saved: training_results.json")
print(f"\n🎉 DONE! You now have a fine-tuned resume matching model.")
